# NLI classifier + synthetic misrepresentations

The NLI classifier (`results/nli_classifier_v1.md`) is the best system so far (test accuracy 0.680,
macro-F1 0.575), but it is still weak at spotting citations that **misrepresent** the paper (NOT
ACCURATE F1 0.44): train has only about 330 real examples of that.

This run makes more. `scripts/generate_misrep.py` has Qwen2.5-7B-Instruct rewrite each accurate
training citation, minimally, so that it misrepresents the same evidence in one way (overstated,
association made causal, overgeneralised, wrong detail, contradicted, or claiming something
unsupported). The NLI model is then trained on real + synthetic data (`train_nli.py --synthetic`).
Test and dev are never touched by the synthetic data.

- **Phase 0** (about 30 to 45 minutes): generate about 1,100 synthetic citations.
- **Phase 1** (about 1.5 hours): 3-fold cross-validation, compared with the NLI model without
  synthetic data (CV macro-F1 0.570). In each fold, only synthetic rows made from training-fold
  citations are used.
- **Phase 2** (about 2 hours): 3 seeds, predicting test. **It runs only if Phase 1 beats 0.570.**

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow
Google Drive access. Everything is saved to `MyDrive/CiteAudit_runs/v6`; if Colab disconnects, Run
all again and finished work is skipped (generation resumes where it stopped).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/v6'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19" sentencepiece protobuf bitsandbytes accelerate

import subprocess, json, statistics
def run(cmd):
    noise = ('UNEXPECTED', 'MISSING', 'LOAD REPORT', 'Key ', '----', 'Notes', '- UNEXP', '- MISS',
             'Loading weights', 'Warning', 'warn(')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

T = 'data_train/target'
SYN = f'{RES}/misrep.jsonl'
def train(name, seed, extra):
    out = f'{RES}/{name}'
    if os.path.exists(f'{out}/config.json'):
        print(f'{name}: already done'); return
    run(f'python scripts/train_nli.py --train {T}/ours_20r_train_cv --dev {T}/ours_20r '
        f'--epochs 3 --batch 2 --accum 4 --seed {seed} --amp --synthetic {SYN} '
        f'--out checkpoints/{name} --pred-dir {out}.part {extra}')
    run(f'rm -rf checkpoints/{name}')
    os.rename(f'{out}.part', out)

LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(rows):
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1
read = lambda p: [json.loads(l) for l in open(p)]
COLS = ['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1']

## Phase 0: generate synthetic misrepresenting citations

In [ ]:
from collections import Counter
if not os.path.exists(f'{RES}/misrep.done'):
    run(f'python scripts/generate_misrep.py --model Qwen/Qwen2.5-7B-Instruct --load-4bit '
        f'--batch 16 --out {SYN}')
    open(f'{RES}/misrep.done', 'w').close()
syn = read(SYN)
print(len(syn), 'synthetic citations;', dict(Counter(r['type'] for r in syn)))
import random
for r in random.Random(1).sample(syn, 8):
    print(f"\n[{r['type']}]\n  original: {r['original']}\n  rewrite:  {r['statement']}")
import torch, gc; gc.collect(); torch.cuda.empty_cache()

## Phase 1: 3-fold cross-validation over train, with synthetic data

In [ ]:
for k in range(3):
    train(f'cv_syn_f{k}', 0, f'--folds 3 --fold {k}')

oof = sum((read(f'{RES}/cv_syn_f{k}/oof.jsonl') for k in range(3)), [])
new = scores(oof)
BASE = [0.676, 0.570, 0.783, 0.513, 0.415]   # NLI without synthetic data, same folds (nli_classifier_v1.md)
print(f'{"cross-validated, 2141 citations":32s} ' + ' '.join(f'{c:>9s}' for c in COLS))
print(f'{"NLI, real data only":32s} ' + ' '.join(f'{x:9.3f}' for x in BASE))
print(f'{"NLI, real + synthetic":32s} ' + ' '.join(f'{x:9.3f}' for x in new))
print('per-fold macro-F1:', [round(scores(read(f'{RES}/cv_syn_f{k}/oof.jsonl'))[1], 3) for k in range(3)])
GO = new[1] > BASE[1]
json.dump({'syn_cv': new, 'nli_cv': BASE, 'phase2': GO}, open(f'{RES}/cv_summary.json', 'w'), indent=2)
print('\nPhase 2 will run.' if GO else '\nSynthetic data does not help; Phase 2 is skipped.')

## Phase 2: 3 seeds on all of train, predicting test (only if Phase 1 won)

In [ ]:
if GO:
    for seed in (0, 1, 2):
        train(f'final_syn_s{seed}', seed, f'--predict test={T}/ours_20r_test')
    runs = [scores(read(f'{RES}/final_syn_s{s}/ours_20r_test.jsonl')) for s in (0, 1, 2)]
    print('test, mean ± sd over 3 seeds')
    for name, xs in zip(COLS, zip(*runs)):
        print(f'  {name:9s} {statistics.mean(xs):.3f} ± {statistics.stdev(xs):.3f}')
    print('NLI without synthetic data on test: accuracy 0.668 ± 0.015, macro-F1 0.556 ± 0.015')

## Download

Zips `MyDrive/CiteAudit_runs/v6` (synthetic data, predictions and logs, no weights). Put `v6.zip`
in the repository root and tell Claude.

In [ ]:
!cd /content/drive/MyDrive/CiteAudit_runs && zip -qr /content/v6.zip v6 -x '*.part/*'
from google.colab import files
files.download('/content/v6.zip')